###**Programación Concurrente**
####Actividad Práctica (Opcional) - Procesos Pesados

---



##**Ejercicio 1 - *Java***

Para compilar y ejecutar codigo en Java en Google Colab:

1.  **Instalar Java Development Kit (JDK)**
2.  **Guardar el codigo de Java como un archivo `.java`**
3.  **Compilar el codigo Java**: Usar el comando `javac`.
4.  **Correr el programa**: Usar el comando `java`.

In [ ]:
!apt-get update
!apt-get install openjdk-11-jdk-headless -qq > /dev/null

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
!java -version

Get:1 https://cli.github.com/packages stable InRelease [4,685 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Hit:3 http://archive.ubuntu.com/ubuntu noble InRelease
Get:4 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:5 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:6 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [72.8 kB]
Get:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Get:10 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.2 MB]
Get:11 http://archive.ubuntu.com/ubuntu noble-updates/restricted amd64 Packages [1,913 kB]
Get:12 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 Packages [2,152 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [2,995 kB]
Get:14

In [ ]:
%%writefile Main.java
import java.io.IOException;

public class Main {
    private static final int TREE_LIFETIME_MS = 60_000;
    public static void main(String[] args) {
        ProcessBuilder processBuilderB = new ProcessBuilder("java", "ChildProcess.java", "B");
        processBuilderB.redirectErrorStream(true);
        processBuilderB.inheritIO();

        try {
            System.out.println(getInfoProcess());

            Process processB = processBuilderB.start();
            processB.waitFor();

            Thread.sleep(TREE_LIFETIME_MS); // Pausa de 60s para poder ver el pstree
        }
        catch (IOException | InterruptedException ex) {
            System.out.println("Error en Proceso A: " + ex.getMessage());
        }
    }

    public static String getInfoProcess() {
        ProcessHandle processHandle = ProcessHandle.current();
        long pid = processHandle.pid();
        long ppid = processHandle.parent().isPresent() ? processHandle.parent().get().pid() : 1;
        return String.format("Proceso A | PID: %s | PPID: %s", pid, ppid);
    }
}

Writing Main.java


In [ ]:
%%writefile ChildProcess.java
import java.io.IOException;

public class ChildProcess {
    private static final int TREE_LIFETIME_MS = 60_000;
    public static void main(String[] args) {
        if (args.length == 0) return;
        String processName = args[0];

        System.out.println("Proceso " + processName + " | " + getInfoProcess());

        // Lógica exacta según el diagrama de tu práctica
        switch (processName) {
            case "B":
                createChild("C", "D");
                break;
            case "C":
                createChild("E");
                break;
            case "D":
                createChild("F", "G");
                break;
            case "E":
                createChild("H", "I");
                break;
            default:
                // F, G, H, I son hojas, no crean hijos
                break;
        }

        // Pausa larga para mantener vivo el árbol y permitir que pstree lo capture
        try {
            Thread.sleep(TREE_LIFETIME_MS);
        } catch (InterruptedException ex) {
            System.out.println("Error en sleep: " + ex.getMessage());
        }
    }

    public static String getInfoProcess() {
        ProcessHandle processHandle = ProcessHandle.current();
        long pid = processHandle.pid();
        long ppid = processHandle.parent().isPresent() ? processHandle.parent().get().pid() : 1;
        return String.format("PID: %s | PPID: %s", pid, ppid);
    }

    private static void createChild(String... hijos) {
        try {
            // Eliminamos p.waitFor() para que los padres no esperen a los hijos,
            // permitiendo que el árbol completo se capture de forma concurrente.
            for (String hijo : hijos) {
                ProcessBuilder pb = new ProcessBuilder("java", "ChildProcess.java", hijo);
                pb.redirectErrorStream(true);
                pb.inheritIO();
                pb.start(); // Creación concurrente (asíncrona)
            }
        } catch (IOException ex) {
            System.out.println("Error al crear hijos: " + ex.getMessage());
        }
    }
}

Writing ChildProcess.java


In [ ]:
%%bash
# 1. Limpiamos procesos anteriores por si quedaron colgados
pkill -f Main.java || true
pkill -f ChildProcess.java || true

# 2. Compilamos todo de nuevo para asegurar que tome los cambios
javac Main.java ChildProcess.java

# 3. Ejecutamos en segundo plano y guardamos la salida
nohup java Main.java 1>salidaJava 2>/dev/null &

# 4. Esperamos 15 segundos para que nazca el árbol completo de 4 niveles
sleep 15


# 5. Mostramos el árbol completo utilizando pstree
ROOT_PID=$(pgrep -f "java Main.java" | head -n 1)

if [ -n "$ROOT_PID" ]; then
    echo "Árbol completo generado desde el PID raíz: $ROOT_PID"
    pstree -pT "$ROOT_PID"
else
    echo "No se pudo encontrar el PID automáticamente."
fi

[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.000s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Árbol completo generado desde el PID raíz: 2578
java(2578)---java(2602)-+-java(2626)---java(2678)-+-java(2750)
                        |                         `-java(2754)
                        `-java(2630)-+-java(2680)
                                     `-java(2696)


**Concluciones**

Con esta actividad comprobamos la creación y el ciclo de vida de procesos concurrentes. Para lograr el árbol del primer ejercicio, fue clave no esperar la finalización secuencial de cada hijo al crearlo, permitiendo que coexistan en memoria. Al pausar su ejecución, pudimos inspeccionar PID y PPID cómo el SO gestiona las relaciones de parentesco a través de los respectivos PCB.

